# Chunking a dataset for teamwork
## For the example, we create a larger test dataset for testing the chunking logic

In [1]:
import random
from itertools import chain
import pandas as pd
from teamwork import teamwork as tw
import networkx as nx
from itertools import combinations  
import requests 
import numpy as np
from datetime import datetime, date, timedelta

In [2]:
test_df = pd.read_csv('../data/sample_notes.csv')
test_df['date'] = pd.to_datetime(test_df['date'])
test_df['arrive_date'] = pd.to_datetime(test_df['arrive_date'])

In [3]:
import pandas as pd

def duplicate_with_date_shift(df, date_cols, interval, num_copies):
    # Create an empty DataFrame to store the results
    result_df = pd.DataFrame()
    
    # Iterate over the number of copies
    for i in range(num_copies):
        # Create a copy of the original DataFrame
        temp_df = df.copy()
        
        # Shift the date columns by (i * interval)
        for col in date_cols:
            temp_df[col] = temp_df[col] + (i * interval)

        temp_df['id'] = temp_df['id'] * (i + 1)
        
        # Append the modified DataFrame to the result DataFrame
        result_df = pd.concat([result_df, temp_df], ignore_index=True)
    
    return result_df


In [4]:
interval = pd.Timedelta(days=10)
num_copies = 100
df = duplicate_with_date_shift(test_df, ['arrive_date','date'], interval, num_copies)

In [5]:
start_date = df.arrive_date.min()

In [6]:
end_date = start_date + timedelta(days=365*2)

## Filter the test dataset to a specific time window

In [7]:
df = df[df['arrive_date'] <= end_date]

## Optional: Running teamwork on full dataset...

In [8]:
corpus = tw.TeamworkCorpus(df)

Preprocessing data...
Building experience edge list...
Building team edge list...


Building Team Dictionary:   0%|          | 0/1056 [00:00<?, ?it/s]

100%|██████████| 139/139 [00:00<00:00, 1138.82it/s]


## Test chunking logic

In [9]:
import pandas as pd
from datetime import timedelta

def iterate_chunks(df, datetime_col, start_date, end_date, lookback_window, target_window):
    current_start = start_date
    chunks = [] 
    while (current_start + lookback_window + target_window) < end_date:
        current_end = current_start + lookback_window + target_window
        buffered_start = current_start - timedelta(days=2)
        buffered_end = current_end + timedelta(days=2)
        chunk = df[(df[datetime_col] >= buffered_start) & (df[datetime_col] < buffered_end)]
        
        if not chunk.empty:
            chunks.append(chunk)
        
        current_start += target_window

    return chunks


In [10]:
lookback_window = timedelta(days=90)
target_window = timedelta(days=10)
df.sort_values('arrive_date', inplace=True)
chunks = iterate_chunks(df, 'arrive_date', start_date, end_date, lookback_window, target_window)
len(chunks)

63

## Run teamwork on chunks

In [11]:
corpuses = dict()
for i, chunk in enumerate(chunks): 
    print(chunk.arrive_date.min())
    print(chunk.date.max())
    corpus_chunk = tw.TeamworkCorpus(chunk)
    corpuses[i] = corpus_chunk

2019-01-01 00:00:00
2019-04-11 19:15:00
Preprocessing data...
Building experience edge list...
Building team edge list...


100%|██████████| 2/2 [00:00<00:00, 1180.66it/s]


2019-01-11 00:00:00
2019-04-21 19:15:00
Preprocessing data...
Building experience edge list...
Building team edge list...


100%|██████████| 3/3 [00:00<00:00, 1343.18it/s]


2019-01-21 00:00:00
2019-05-01 19:15:00
Preprocessing data...
Building experience edge list...
Building team edge list...


...

100%|██████████| 4/4 [00:00<00:00, 1212.84it/s]


2020-09-02 00:00:00
2020-12-11 19:15:00
Preprocessing data...
Building experience edge list...
Building team edge list...


100%|██████████| 3/3 [00:00<00:00, 1099.81it/s]


2020-09-12 00:00:00
2020-12-21 19:15:00
Preprocessing data...
Building experience edge list...
Building team edge list...


100%|██████████| 3/3 [00:00<00:00, 358.51it/s]


## Prepare function for calculating team experience

In [12]:
def get_output_for_row(g, visit_id, team):
    data = {}
    data['visit_id'] = visit_id
    
    ''' Clustering coefficient of all nodes (in a dictionary) '''
    clustering_coefficient = nx.clustering(g, weight='weight')
    
    ''' Average clustering coefficient with divide-by-zero check '''
    clust_sum = sum(clustering_coefficient.values())
    clust_len = len(clustering_coefficient)
        
    data['avg_clust'] = clust_sum / clust_len if clust_len > 0 else 0
    
    data['sum_clust'] = clust_sum
    
    data['team_size'] = len(team)
    potential_edges = len(list(combinations(team,2)))
    data['potential_edges'] = potential_edges
    data['team_edge_size'] = g.number_of_edges()
    
    experience = g.size(weight='weight') #Experience as sum of weights
    data['experience'] = experience
    
    data['cumulative_experience'] = experience - data['team_edge_size']
    
    data['avg_cumulative_experience'] = data['cumulative_experience'] / potential_edges if data['team_size'] > 0 else 0
    
    return data
    
    

## Merge chunks into a dictionary

In [13]:
merged_visits = dict()
for i, chunked_corpus in corpuses.items():
    for visit_id, item in chunked_corpus.team_experience_dict.items():
        merged_visits[visit_id] = item


## Gather output into a DataFrame

In [14]:
output = [get_output_for_row(item['graph'], visit_id, item['team']) for visit_id, item in merged_visits.items()]
teamwork_results_df = pd.DataFrame.from_dict(output, orient='columns')